# Chapter 6.2: GAN-based Recommendation

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Understand** the adversarial training framework and its application to recommendation
2. **Implement** the IRGAN model combining generative and discriminative retrieval
3. **Build** CFGAN for generating synthetic user-item interactions
4. **Apply** adversarial training for data augmentation in recommendation systems
5. **Diagnose** mode collapse and training instability in GAN-based recommenders
6. **Compare** GAN-based methods against VAE-based collaborative filtering

## Prerequisites

- Familiarity with PyTorch and neural network training
- Understanding of GANs (generator, discriminator, adversarial loss)
- Knowledge of collaborative filtering basics (Part 1)
- Chapter 6.1 (VAE for CF) for comparison

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USERNAME/rec_system/blob/main/notebooks/part6/chapter_6.2_gan_rec.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/USERNAME/rec_system/main/notebooks/part6/chapter_6.2_gan_rec.ipynb)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from collections import defaultdict

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cpu')
print(f'Using device: {device}')

## 1. Background: GANs for Information Retrieval

### IRGAN: Unifying Generative and Discriminative Models

**IRGAN** (Wang et al., SIGIR 2017) was the first work to apply GANs to information retrieval and recommendation. The key insight is:

- **Discriminative model** $D(u, i)$: Estimates whether item $i$ is relevant to user $u$ based on observed data
- **Generative model** $G(i|u)$: Generates (samples) items for user $u$ that look like real interactions

The minimax game:
$$\min_G \max_D \sum_{u} \left[ \sum_{i \in \mathcal{R}_u} \log D(u, i) + \sum_{i \sim G(\cdot|u)} \log(1 - D(u, i)) \right]$$

where $\mathcal{R}_u$ is user $u$'s positive interactions.

### Why GANs for Recommendation?

1. **Data augmentation**: Generate synthetic interactions to alleviate sparsity
2. **Hard negative sampling**: The generator learns to produce challenging negatives
3. **Distribution matching**: Align generated distributions with real user behavior

> **💡 Concept:** Unlike image GANs that generate pixels, recommendation GANs typically generate discrete items or interaction vectors. This creates unique challenges for gradient-based training.

In [ ]:
def generate_synthetic_data(n_users=1500, n_items=400, n_factors=8, sparsity=0.05, seed=42):
    """Generate synthetic implicit feedback data."""
    rng = np.random.RandomState(seed)
    user_factors = rng.randn(n_users, n_factors) * 0.5
    item_factors = rng.randn(n_items, n_factors) * 0.5
    item_bias = rng.randn(n_items) * 0.3
    
    logits = user_factors @ item_factors.T + item_bias[None, :]
    probs = 1.0 / (1.0 + np.exp(-logits))
    threshold = np.quantile(probs, 1 - sparsity)
    interactions = (probs > threshold).astype(np.float32)
    
    train_data = interactions.copy()
    test_data = np.zeros_like(interactions)
    for u in range(n_users):
        items = np.where(interactions[u] > 0)[0]
        if len(items) > 1:
            n_test = max(1, int(len(items) * 0.2))
            test_items = rng.choice(items, n_test, replace=False)
            train_data[u, test_items] = 0
            test_data[u, test_items] = 1
    
    return train_data, test_data

train_data, test_data = generate_synthetic_data()
n_users, n_items = train_data.shape
print(f'Users: {n_users}, Items: {n_items}')
print(f'Train interactions: {train_data.sum():.0f} (density: {train_data.mean():.4f})')
print(f'Test interactions: {test_data.sum():.0f}')

## 2. IRGAN Implementation

IRGAN uses REINFORCE to train the generator since sampling items is a discrete operation.

The generator policy:
$$G(i|u) = \frac{\exp(\mathbf{g}_u^\top \mathbf{g}_i)}{\sum_{j \notin \mathcal{R}_u} \exp(\mathbf{g}_u^\top \mathbf{g}_j)}$$

> **⚠️ Common Pitfall:** IRGAN is notoriously hard to train. The generator often collapses to sampling only a few popular items. Use learning rate warmup and gradient clipping.

In [ ]:
class IRGANDiscriminator(nn.Module):
    """Discriminator for IRGAN: predicts relevance of (user, item) pairs."""
    
    def __init__(self, n_users, n_items, embed_dim=32):
        super().__init__()
        self.user_embed = nn.Embedding(n_users, embed_dim)
        self.item_embed = nn.Embedding(n_items, embed_dim)
        nn.init.normal_(self.user_embed.weight, std=0.01)
        nn.init.normal_(self.item_embed.weight, std=0.01)
    
    def forward(self, user_ids, item_ids):
        u = self.user_embed(user_ids)
        i = self.item_embed(item_ids)
        return torch.sigmoid(torch.sum(u * i, dim=-1))


class IRGANGenerator(nn.Module):
    """Generator for IRGAN: samples items for users."""
    
    def __init__(self, n_users, n_items, embed_dim=32):
        super().__init__()
        self.n_items = n_items
        self.user_embed = nn.Embedding(n_users, embed_dim)
        self.item_embed = nn.Embedding(n_items, embed_dim)
        nn.init.normal_(self.user_embed.weight, std=0.01)
        nn.init.normal_(self.item_embed.weight, std=0.01)
    
    def get_scores(self, user_ids):
        """Get item scores for given users."""
        u = self.user_embed(user_ids)  # (batch, dim)
        all_items = self.item_embed.weight  # (n_items, dim)
        return u @ all_items.T  # (batch, n_items)
    
    def sample(self, user_ids, mask, n_samples=1, temperature=1.0):
        """Sample items for users using the generator policy."""
        scores = self.get_scores(user_ids) / temperature
        # Mask out positive items
        scores = scores.masked_fill(mask.bool(), -1e9)
        probs = F.softmax(scores, dim=-1)
        sampled = torch.multinomial(probs, n_samples, replacement=False)
        return sampled, probs


def train_irgan(train_data, test_data, n_epochs=30, embed_dim=32,
                g_lr=1e-3, d_lr=1e-3, n_gen_samples=5, d_steps=3, g_steps=1):
    """Train IRGAN with alternating generator/discriminator updates."""
    n_users, n_items = train_data.shape
    
    gen = IRGANGenerator(n_users, n_items, embed_dim).to(device)
    disc = IRGANDiscriminator(n_users, n_items, embed_dim).to(device)
    
    g_opt = torch.optim.Adam(gen.parameters(), lr=g_lr)
    d_opt = torch.optim.Adam(disc.parameters(), lr=d_lr)
    
    train_tensor = torch.FloatTensor(train_data).to(device)
    user_ids_all = torch.arange(n_users).to(device)
    
    history = {'d_loss': [], 'g_loss': [], 'recall': [], 'ndcg': []}
    
    for epoch in range(n_epochs):
        # --- Train Discriminator ---
        disc.train()
        gen.eval()
        d_losses = []
        
        for _ in range(d_steps):
            batch_users = torch.randint(0, n_users, (256,)).to(device)
            
            # Positive samples
            pos_items = []
            for u in batch_users.cpu().numpy():
                pos = np.where(train_data[u] > 0)[0]
                if len(pos) > 0:
                    pos_items.append(np.random.choice(pos))
                else:
                    pos_items.append(0)
            pos_items = torch.LongTensor(pos_items).to(device)
            
            # Negative samples from generator
            with torch.no_grad():
                neg_items, _ = gen.sample(batch_users, train_tensor[batch_users], n_samples=1)
                neg_items = neg_items.squeeze(-1)
            
            pos_scores = disc(batch_users, pos_items)
            neg_scores = disc(batch_users, neg_items)
            
            d_loss = -torch.mean(torch.log(pos_scores + 1e-8) + torch.log(1 - neg_scores + 1e-8))
            
            d_opt.zero_grad()
            d_loss.backward()
            torch.nn.utils.clip_grad_norm_(disc.parameters(), 1.0)
            d_opt.step()
            d_losses.append(d_loss.item())
        
        # --- Train Generator (REINFORCE) ---
        gen.train()
        disc.eval()
        g_losses = []
        
        for _ in range(g_steps):
            batch_users = torch.randint(0, n_users, (256,)).to(device)
            
            sampled_items, probs = gen.sample(batch_users, train_tensor[batch_users],
                                              n_samples=n_gen_samples)
            
            # Reward from discriminator
            with torch.no_grad():
                rewards = []
                for j in range(n_gen_samples):
                    r = disc(batch_users, sampled_items[:, j])
                    rewards.append(torch.log(r + 1e-8))
                rewards = torch.stack(rewards, dim=1)  # (batch, n_samples)
            
            # REINFORCE loss
            log_probs = []
            for j in range(n_gen_samples):
                lp = torch.log(probs.gather(1, sampled_items[:, j:j+1]) + 1e-8).squeeze(-1)
                log_probs.append(lp)
            log_probs = torch.stack(log_probs, dim=1)
            
            # Baseline: mean reward
            baseline = rewards.mean(dim=1, keepdim=True)
            g_loss = -torch.mean(log_probs * (rewards - baseline))
            
            g_opt.zero_grad()
            g_loss.backward()
            torch.nn.utils.clip_grad_norm_(gen.parameters(), 1.0)
            g_opt.step()
            g_losses.append(g_loss.item())
        
        # Evaluate
        if (epoch + 1) % 5 == 0:
            disc.eval()
            with torch.no_grad():
                all_scores = disc.user_embed.weight @ disc.item_embed.weight.T
                all_scores[train_tensor > 0] = -float('inf')
                _, top_k = torch.topk(all_scores, 20, dim=1)
                top_k = top_k.cpu().numpy()
            
            recalls, ndcgs = [], []
            for u in range(n_users):
                true = set(np.where(test_data[u] > 0)[0])
                if len(true) == 0:
                    continue
                hits = [1 if i in true else 0 for i in top_k[u]]
                recalls.append(sum(hits) / min(len(true), 20))
                dcg = sum(h / np.log2(j + 2) for j, h in enumerate(hits))
                idcg = sum(1.0 / np.log2(j + 2) for j in range(min(len(true), 20)))
                ndcgs.append(dcg / idcg if idcg > 0 else 0)
            
            avg_recall = np.mean(recalls)
            avg_ndcg = np.mean(ndcgs)
            history['d_loss'].append(np.mean(d_losses))
            history['g_loss'].append(np.mean(g_losses))
            history['recall'].append(avg_recall)
            history['ndcg'].append(avg_ndcg)
            print(f'Epoch {epoch+1:3d} | D_loss: {np.mean(d_losses):.4f} | '
                  f'G_loss: {np.mean(g_losses):.4f} | '
                  f'Recall@20: {avg_recall:.4f} | NDCG@20: {avg_ndcg:.4f}')
    
    return gen, disc, history


gen, disc, irgan_history = train_irgan(train_data, test_data, n_epochs=30)

In [ ]:
# Plot IRGAN training
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs = list(range(5, 31, 5))

axes[0].plot(epochs, irgan_history['d_loss'], 'b-o', label='Discriminator Loss', markersize=6)
axes[0].plot(epochs, irgan_history['g_loss'], 'r-s', label='Generator Loss', markersize=6)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('IRGAN Training Losses')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, irgan_history['recall'], 'g-o', label='Recall@20', markersize=6)
axes[1].plot(epochs, irgan_history['ndcg'], 'm-s', label='NDCG@20', markersize=6)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Metric')
axes[1].set_title('IRGAN Recommendation Quality')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. CFGAN: Collaborative Filtering with GANs

**CFGAN** (Chae et al., CIKM 2018) takes a different approach: instead of sampling items, the generator directly produces a continuous interaction vector that mimics the user's real interaction pattern.

Architecture:
- **Generator** $G(\mathbf{c}, \mathbf{z})$: Takes a condition vector $\mathbf{c}$ (partial interactions) and noise $\mathbf{z}$, outputs a fake interaction vector
- **Discriminator** $D(\mathbf{x})$: Distinguishes real interaction vectors from generated ones

Key innovations:
1. Condition on partial user history (masking)
2. Zero-reconstruction regularization: only reconstruct observed items

> **💡 Concept:** CFGAN works in the vector space of item interactions, avoiding the discrete sampling problem of IRGAN. This makes it more stable to train.

In [ ]:
class CFGANGenerator(nn.Module):
    """CFGAN Generator: generates interaction vectors conditioned on partial history.
    
    Reference: Chae et al., "CFGAN: A Generic CF Framework based on GANs", CIKM 2018.
    """
    
    def __init__(self, n_items, noise_dim=32, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_items + noise_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(hidden_dim),
            nn.Linear(hidden_dim, n_items),
            nn.Sigmoid()
        )
        self.noise_dim = noise_dim
    
    def forward(self, condition, noise=None):
        if noise is None:
            noise = torch.randn(condition.size(0), self.noise_dim).to(condition.device)
        x = torch.cat([condition, noise], dim=1)
        return self.net(x)


class CFGANDiscriminator(nn.Module):
    """CFGAN Discriminator: distinguishes real vs generated interaction vectors."""
    
    def __init__(self, n_items, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_items, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_cfgan(train_data, test_data, n_epochs=60, batch_size=256,
                g_lr=2e-4, d_lr=2e-4, noise_dim=32, mask_ratio=0.3,
                recon_weight=0.1):
    """Train CFGAN with zero-reconstruction regularization."""
    n_users, n_items = train_data.shape
    
    gen = CFGANGenerator(n_items, noise_dim=noise_dim).to(device)
    disc = CFGANDiscriminator(n_items).to(device)
    
    g_opt = torch.optim.Adam(gen.parameters(), lr=g_lr, betas=(0.5, 0.999))
    d_opt = torch.optim.Adam(disc.parameters(), lr=d_lr, betas=(0.5, 0.999))
    
    train_tensor = torch.FloatTensor(train_data).to(device)
    dataset = TensorDataset(train_tensor)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    history = {'d_loss': [], 'g_loss': [], 'recall': [], 'ndcg': []}
    
    for epoch in range(n_epochs):
        gen.train()
        disc.train()
        epoch_d_loss = 0
        epoch_g_loss = 0
        n_batches = 0
        
        for (real_x,) in loader:
            batch_size_actual = real_x.size(0)
            
            # Create condition by masking some interactions
            mask = (torch.rand_like(real_x) > mask_ratio).float()
            condition = real_x * mask
            
            # --- Train Discriminator ---
            noise = torch.randn(batch_size_actual, noise_dim).to(device)
            fake_x = gen(condition, noise).detach()
            
            real_pred = disc(real_x)
            fake_pred = disc(fake_x)
            
            d_loss = -torch.mean(torch.log(real_pred + 1e-8) + torch.log(1 - fake_pred + 1e-8))
            
            d_opt.zero_grad()
            d_loss.backward()
            d_opt.step()
            
            # --- Train Generator ---
            noise = torch.randn(batch_size_actual, noise_dim).to(device)
            fake_x = gen(condition, noise)
            fake_pred = disc(fake_x)
            
            # Adversarial loss
            g_adv_loss = -torch.mean(torch.log(fake_pred + 1e-8))
            
            # Zero-reconstruction: generated values for observed items should match
            recon_loss = F.mse_loss(fake_x * real_x, real_x)
            
            g_loss = g_adv_loss + recon_weight * recon_loss
            
            g_opt.zero_grad()
            g_loss.backward()
            g_opt.step()
            
            epoch_d_loss += d_loss.item()
            epoch_g_loss += g_loss.item()
            n_batches += 1
        
        # Evaluate
        if (epoch + 1) % 10 == 0:
            gen.eval()
            with torch.no_grad():
                # Use generator to predict scores
                scores = gen(train_tensor)
                scores[train_tensor > 0] = -float('inf')
                _, top_k = torch.topk(scores, 20, dim=1)
                top_k = top_k.cpu().numpy()
            
            recalls, ndcgs = [], []
            for u in range(n_users):
                true = set(np.where(test_data[u] > 0)[0])
                if len(true) == 0:
                    continue
                hits = [1 if i in true else 0 for i in top_k[u]]
                recalls.append(sum(hits) / min(len(true), 20))
                dcg = sum(h / np.log2(j + 2) for j, h in enumerate(hits))
                idcg = sum(1.0 / np.log2(j + 2) for j in range(min(len(true), 20)))
                ndcgs.append(dcg / idcg if idcg > 0 else 0)
            
            r = np.mean(recalls)
            n = np.mean(ndcgs)
            history['d_loss'].append(epoch_d_loss / n_batches)
            history['g_loss'].append(epoch_g_loss / n_batches)
            history['recall'].append(r)
            history['ndcg'].append(n)
            print(f'CFGAN Epoch {epoch+1:3d} | D: {epoch_d_loss/n_batches:.4f} | '
                  f'G: {epoch_g_loss/n_batches:.4f} | Recall@20: {r:.4f} | NDCG@20: {n:.4f}')
    
    return gen, disc, history


cfgan_gen, cfgan_disc, cfgan_history = train_cfgan(train_data, test_data, n_epochs=60)

## 4. Adversarial Data Augmentation

Beyond end-to-end GAN models, adversarial training can augment sparse interaction data:

1. Train a generator to produce plausible interactions
2. Add generated interactions to the training set
3. Re-train the recommendation model on the augmented data

This approach is simpler and often more stable than full GAN training.

In [ ]:
def augment_with_generator(gen, train_data, threshold=0.5, max_new_per_user=10):
    """Use trained generator to augment training data."""
    gen.eval()
    train_tensor = torch.FloatTensor(train_data).to(device)
    
    with torch.no_grad():
        generated = gen(train_tensor).cpu().numpy()
    
    augmented = train_data.copy()
    n_added = 0
    
    for u in range(len(train_data)):
        # Find items not in training set with high generated scores
        candidates = np.where((train_data[u] == 0) & (generated[u] > threshold))[0]
        if len(candidates) > max_new_per_user:
            top_idx = np.argsort(generated[u, candidates])[-max_new_per_user:]
            candidates = candidates[top_idx]
        augmented[u, candidates] = 1
        n_added += len(candidates)
    
    return augmented, n_added


augmented_data, n_added = augment_with_generator(cfgan_gen, train_data, threshold=0.5)
print(f'Original interactions: {train_data.sum():.0f}')
print(f'Added interactions: {n_added}')
print(f'Augmented interactions: {augmented_data.sum():.0f}')
print(f'Density change: {train_data.mean():.4f} -> {augmented_data.mean():.4f}')

## 5. Mode Collapse Analysis

Mode collapse is a critical issue in GAN-based recommendation where the generator only produces a small set of popular items.

> **⚠️ Common Pitfall:** Mode collapse in recommendation GANs manifests as the generator always recommending the same popular items regardless of user. Monitor the diversity of generated items during training.

In [ ]:
def analyze_mode_collapse(gen, train_data, n_samples=500):
    """Analyze whether the generator suffers from mode collapse."""
    gen.eval()
    train_tensor = torch.FloatTensor(train_data[:n_samples]).to(device)
    
    with torch.no_grad():
        generated = gen(train_tensor).cpu().numpy()
    
    # Find top predicted items per user (items not in training)
    all_top_items = []
    for u in range(n_samples):
        scores = generated[u].copy()
        scores[train_data[u] > 0] = -1  # Mask training items
        top_10 = np.argsort(scores)[-10:]
        all_top_items.extend(top_10)
    
    unique_items = len(set(all_top_items))
    total_items = len(all_top_items)
    
    # Item frequency distribution
    item_counts = defaultdict(int)
    for item in all_top_items:
        item_counts[item] += 1
    
    counts = sorted(item_counts.values(), reverse=True)
    
    return {
        'unique_items': unique_items,
        'total_recommendations': total_items,
        'coverage': unique_items / train_data.shape[1],
        'top_item_counts': counts[:20],
        'gini': _gini_coefficient(counts)
    }


def _gini_coefficient(values):
    """Compute Gini coefficient (0 = perfect equality, 1 = max inequality)."""
    values = np.array(sorted(values))
    n = len(values)
    cumsum = np.cumsum(values)
    return (2 * np.sum((np.arange(1, n + 1) * values)) / (n * np.sum(values))) - (n + 1) / n


collapse_stats = analyze_mode_collapse(cfgan_gen, train_data)
print(f"Unique items recommended: {collapse_stats['unique_items']} / {n_items}")
print(f"Item coverage: {collapse_stats['coverage']:.2%}")
print(f"Gini coefficient: {collapse_stats['gini']:.4f} (higher = more collapse)")
print(f"Top 10 item frequencies: {collapse_stats['top_item_counts'][:10]}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(min(50, len(collapse_stats['top_item_counts']))),
            collapse_stats['top_item_counts'][:50], color='steelblue')
axes[0].set_xlabel('Item Rank')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Generated Item Frequency Distribution')
axes[0].grid(True, alpha=0.3)

# Compare IRGAN vs CFGAN metrics
methods = ['IRGAN', 'CFGAN']
final_recalls = [irgan_history['recall'][-1] if irgan_history['recall'] else 0,
                 cfgan_history['recall'][-1] if cfgan_history['recall'] else 0]
final_ndcgs = [irgan_history['ndcg'][-1] if irgan_history['ndcg'] else 0,
               cfgan_history['ndcg'][-1] if cfgan_history['ndcg'] else 0]

x_pos = np.arange(len(methods))
width = 0.35
axes[1].bar(x_pos - width/2, final_recalls, width, label='Recall@20', color='steelblue')
axes[1].bar(x_pos + width/2, final_ndcgs, width, label='NDCG@20', color='coral')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(methods)
axes[1].set_ylabel('Metric Value')
axes[1].set_title('GAN Methods Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Exercises

### 🏋️ Exercise 1: Implement CFGAN with Wasserstein Loss

Replace the standard GAN loss with Wasserstein loss (WGAN-GP) for more stable training.

In [ ]:
# TODO: Implement WGAN-GP for CFGAN
def gradient_penalty(disc, real_data, fake_data, lambda_gp=10):
    """
    TODO: Implement gradient penalty
    1. Sample alpha uniformly from [0, 1]
    2. Create interpolated samples: x_hat = alpha * real + (1 - alpha) * fake
    3. Compute discriminator output on interpolated samples
    4. Compute gradient of output w.r.t. x_hat
    5. Penalty = lambda * (||grad|| - 1)^2
    """
    pass

# TODO: Modify the training loop to use Wasserstein loss:
# D_loss = D(fake) - D(real) + gradient_penalty
# G_loss = -D(fake)
# Note: Remove sigmoid from discriminator output

### 🏋️ Exercise 2: Implement Diversity-Promoting Generator

Add a diversity regularization term to the generator loss to combat mode collapse.

In [ ]:
# TODO: Implement diversity-aware generator training
def diversity_loss(generated_vectors, lambda_div=0.01):
    """
    TODO: Compute diversity regularization
    Hint: Penalize high cosine similarity between generated vectors of different users
    1. Normalize generated vectors
    2. Compute pairwise cosine similarity matrix
    3. Return mean off-diagonal similarity (to minimize)
    """
    pass

# TODO: Add diversity_loss to generator training and compare mode collapse metrics

### 🏋️ Exercise 3: Compare GAN vs VAE for CF

Run a fair comparison between CFGAN and Mult-VAE on the same data.

In [ ]:
# TODO: Fair comparison
# 1. Train Mult-VAE (from Chapter 6.1) with same hidden_dim and latent_dim
# 2. Train CFGAN with same hidden_dim
# 3. Compare:
#    - Recall@20 and NDCG@20
#    - Training time
#    - Item coverage (diversity)
#    - Training stability (loss variance)
# 4. Plot all comparisons

## Summary

| Model | Approach | Key Innovation | Challenge |
|-------|----------|---------------|----------|
| **IRGAN** (2017) | Generative + Discriminative IR | REINFORCE for discrete sampling | Training instability |
| **CFGAN** (2018) | Continuous vector generation | Zero-reconstruction regularization | Mode collapse |
| **Adversarial Augmentation** | Data augmentation | Simple and modular | Quality control |

**Key Takeaways:**
1. GAN-based recommendation is powerful but harder to train than VAE-based methods
2. Mode collapse is a persistent challenge — monitor item diversity
3. CFGAN's continuous approach is more stable than IRGAN's discrete sampling
4. Adversarial data augmentation is often more practical than end-to-end GAN training